## Use of LLM Components as Runnables (without LLMChain class)
### Simple use case
- Using LLMs which were in use at starting before ChatModels
- Simple LLM chain without LLMChain class

In [ ]:
from langchain_openai import OpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

# required to load api_key to use OpenAI api
load_dotenv()


# initialize the LLM
llm = OpenAI(
    model_name='gpt-3.5-turbo', 
    temperature=0.7
    )


# create a prompt template
prompt = PromptTemplate(
    input_variables = ["topic"],
    template = "Suggest a catchy blog title about {topic}"
)

# define the input
topic = input('Enter a topic')

# format the prompt manually using PromptTemplate
formatted_prompt = prompt.format(topic=topic)

# call the LLM directly
blog_title = llm.predict(formatted_prompt)

# print the output
print("Generated Blog Title:", blog_title)

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### PDF Reader Usecase (Without LLMChain class)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import FAISS

# load the document
loader = TextLoader("docs.txt")  # ensure docs.txt exists
documents = loader.load()

# Split the text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Convert text into embeddings & store in FAISS
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())

# Create a retriever (fetches relevant documents)
retriever = vectorstore.as_retriever()

# Manually Retrieve Relevant Documents
query = "What are the key takeaways from the document?"
retrieved_docs = retriever.get_relevant_documents(query)

# Combine Retrieved Text into a Single Prompt
retrieved_text = "\n".join([doc.page_content for doc in retrieved_docs])

# Initialize the LLM
llm = OpenAI(
    model_name="gpt-3.5-turbo", 
    temperature=0.7
    )

# Manually Pass Retrieved Text to LLM
prompt = f"Based on the following text, answer the question: {query}\n\n {retrieved_text}"
answer = llm.predict(prompt)

# print the answer
print("Answer:", answer)

## Via LLMChain Class
- This `LLMChain` class is `deprecated in current version`. So, we can't see output from it, but we can see how it has made life easier.
- Apart from `LLMChain`, rest libraries are according to current version, only import `from langchain.chains import LLMChain` is used from previous version, this is not an issue here, as we are just referring athis code snippet to understand why and how `chains` are useful.
### Simple One (First Example Above)
- First we'll see first simple example example

In [ ]:
from langchain_openai import OpenAI
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate


load_dotenv() 

llm = OpenAI(
    model_name='gpt-3.5-turbo', 
    temperature=0.7
    )

prompt = PromptTemplate(
    input_variables = ["topic"],
    template = "Suggest a catchy blog title about {topic}"
)

# create an LLM Chain
chain = LLMChain(llm=llm, prompt=prompt)

# run the chain with aspecific topic
topic = input('enter a topic')
output = chain.run(topic)

# print the output
print("Generated Blog Title:", output)

- Here no manual work of calling each part seperately as earlier example.
- Only LLMChain class should be called with llm and prompt, all steps will be happening internally

### PDFReader Example
- Here let's see usage of `RetrievalQA` in the pdf reader example.
- This is a RAG example, so `RetrievalQA` is used instead of `LLMChain`.
- Here also, this code will not run as `from langchain.chains import RetrievalQA` is from previous version and in current version this is deprecated.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# load the document
loader = TextLoader("docs.txt")  # ensure docs.txt exists
documents = loader.load()

# Split the text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Convert text into embeddings & store in FAISS
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())

# Create a retriever (fetches relevant documents)
retriever = vectorstore.as_retriever()

# Initialize the LLM
llm = OpenAI(
    model_name="gpt-3.5-turbo", 
    temperature=0.7
    )

# create RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm = llm, 
    retriever = retriever
    )

# Ask a question
query = "What are the key takeaways from the document?"
answer = qa_chain.run(query)


# print the answer
print("Answer:", answer)

- Here manual heavy-lifting is replaced by `Chains`.

## Problem with Above Approach
- LangChain team had understood that, from starting different components were not written in standardized way - for llm call `predict` method, to get retrieved docs `get_relevant_documents` method, it's not flexible.
- We will see basic concept of implemntation of components in LangChain and why it's not standardized.

In [12]:
# langchain aam zindegi
import random 

class NakliLLM:
    def __init__(self):
        print('LLM created')

    def predict(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}

In [13]:
# creating an object
llm = NakliLLM()

LLM created


In [ ]:
# asking a query
llm.predict('What is capital of India?')

{'response': 'AI stands for Artificial Intelligence'}

In [15]:
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):
        return self.template.format(**input_dict)

- See invokation of `NakliPromptTemplate` with one parameter.

In [16]:
template1 = NakliPromptTemplate(
    template = 'Write a poem about {topic}',
    input_variables = ['topic']
)

prompt1 = template1.format({'topic': 'India'})
print(prompt1)

llm1 = NakliLLM()

llm1.predict(prompt1)

Write a poem about India
LLM created


{'response': 'AI stands for Artificial Intelligence'}

- Now see same with two parameters.

In [17]:
template2 = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
)

prompt2 = template1.format({'length':'short', 'topic': 'India'})
print(prompt2)

llm2 = NakliLLM()

llm2.predict(prompt2)

Write a poem about India
LLM created


{'response': 'IPL is a cricket leage'}

In [ ]:
class NakliLLMChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self, input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)
        return result['response']

In [ ]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
)

llm = NakliLLM()

chain = NakliLLMChain(llm, template)
chain.run({'length':'short', 'topic': 'India'})

- Can't do two step llm call by this process. It's not flexible to create any type of workflows.
- So next we will see how runnables works internally to get this standadization.

## Standardization
- Here we'll learn basic concept of implementing runnables in LangChain from scratch to understand how standadization is applied in new version which was not in initial versions.
- We'll see how standardization happened using invoke() method.
- This is solution of above problem.
- To make sure all runnable classes have same methods (which is the `standardization` itself we are talking), `abstraction` is used via abstruct method in python.
- We will make an abstract class, and all later classes will be inherited from it.
- Abstract class concept is OOPs concept which forces child classes to inherit from parent class. Here methods written in parent class is overridden by same methods written in child class.

In [ ]:
from abc import ABC, abstractmethod

class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass


class NakliLLM(Runnable):
    def __init__(self):
        print('LLM created')

    def invoke(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}

    # LangChain team's thought:
    # we'll give a warning here that `predict` method is deprecated otherwise already written codes will start breaking, 
    # bcz those codes will use their environment contains version having `predict` and in any coding env, virtual env got creates and frameworks got downloaded according to mentioned version
    # In new versions gradually we can remove this method  
    def predict(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}


class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self):
        return self.template.format(**self.input_variables)

    def format(self):
        return self.template.format(**self.input_variables)

Note: Here if we'll use the code `llm = NakliLLM()` to initialize `NakliLLM` class, it will throw an error if the `invoke` method is not implemented. This is what we wanted to keep consistency among all classes.

In [ ]:
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)

        return input_data


In [ ]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
)

llm = NakliLLM()

In [ ]:
chain = RunnableConnector([template, llm])
chain.invoke({'length': 'short', 'topic': 'India'})

Let's add outpur parser concept.

In [ ]:
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data['response']

In [ ]:
parser = NakliStrOutputParser()

In [ ]:
chain = RunnableConnector([template, llm, parser])
chain.invoke({'length': 'short', 'topic': 'India'})

In [ ]:
# connecting chains 
template1 = NakliPromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

template2 = NakliPromptTemplate(
    template = 'Explain following joke {response}',
    input_variables = ['response']
)

llm = NakliLLM()
parser = NakliStrOutputParser()

chain1 = RunnableConnector([template1, llm])
chain2 = RunnableConnector([template2, llm, parser])

final_chain = RunnableConnector([chain1, chain2])
final_chain.invoke({'topic': 'cricket'})

## Different Types of Runnables Present in LangChain
- These are equivalent as we seen in `RunnableConnector` above.

### RunnableSequence
- This is used to implement linear chains.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence

In [ ]:
load_dotenv()

In [ ]:
prompt1 = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

prompt2 = PromptTemplate(
    template = 'Explain the following joke - {text}',
    input_variables = ['text']
)

parser = StrOutputParser()

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)

print(chain.invoke({'topic': 'AI'}))

### RunnableParallel
- This is used to implement parallel chains.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a tweet about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Explain the Linkedin joke - {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

parallet_chain = RunnableParallel(
    {
        'tweet': RunnableSequence(prompt1, model, parser),
        'linkedin': RunnableSequence(prompt2, model, parser)
    }
)

result = parallet_chain.invoke({'topic': 'AI'})
print(result)

In [ ]:
print(result['tweet'])
print(result['linkedin'])

### RunnablePassthrough
- This is used when we need output same as input from a Runnable execution.
- E.g. - in below exaple, we want to show joke along with explanation, but for it joke should be generated first and same should be shown as a prallel chain of it's expalnation part. So here as in this chain, nothing to do actually, this is a dummy runnable creation.

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough

load_dotenv()

passthrough = RunnablePassthrough()

print(passthrough.invoke(2))
print(passthrough.invoke({'name': 'koyel'}))


2
{'name': 'koyel'}


In [10]:
prompt1 = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

prompt2 = PromptTemplate(
    template = 'Explain the following joke - {text}',
    input_variables = ['text']
)

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt1, model, parser)
parallel_chain = RunnableParallel(
    {
        'joke': RunnablePassthrough(),
        'explanation': RunnableSequence(prompt2, model, parser)
    }
)

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic': 'cricket'}))

{'joke': 'Why did the cricket team go to the bank?\n\nTo get their bowlers!', 'explanation': 'This joke plays on the double meaning of "bowlers." In cricket, a bowler is a player who delivers the ball to the batsman. However, in the context of the joke, "bowlers" can also refer to a type of hat, typically worn by bankers. So, the joke is a play on words, implying that the cricket team went to the bank to retrieve their bowlers (hats) instead of their bowlers (players).'}


In [11]:
result = final_chain.invoke({'topic': 'cricket'})
print(result['joke'])
print(result['explanation'])

Why did the cricket team go to the bank?

Because they wanted to make a good withdrawal!
This joke is a play on words. In cricket, a 'good' shot is called a 'withdrawal'. So, when the cricket team went to the bank, they wanted to make a 'good withdrawal' in terms of money, but it also sounds like they wanted to make a good shot in the game of cricket.


### RunnableLambda
- This is used to `convert` any `lambda function` to `Runnable`.

#### Simple Example 
- To understand how `RunnableLambda` converts lambda functions to runnable.

In [5]:
from langchain_core.runnables import RunnableLambda

def word_counter(text):
    return len(text.split())

runnable_word_counter = RunnableLambda(word_counter)

print(runnable_word_counter.invoke('Hi there, how are you?'))

5


- As we know lambda function is those function which has only one instruction to perform in it.
- We can define these one work functions using `lambda` key-word, that's why it's called lambda functions.
- See below how instead of general function it can be written using `lambda` key-word and how `RunnableLambda` is used on it to make it a runnable.

In [6]:
from langchain_core.runnables import RunnableLambda

# instead of writing seperate function `word_count`, it can be directly used during RunnableLambda class initialization.
runnable_word_counter = RunnableLambda(lambda x: len(x.split()))

print(runnable_word_counter.invoke('Hi there, how are you?'))

5


#### Use of `RunnableLambda` within Parallel Chain Example Shown Earlier in `RunnableParallel` Section

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()

def word_count(text):
    return len(text.split())

prompt = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt, model, parser)

parallel_chain = RunnableParallel(
    {
        'joke': RunnablePassthrough(),
        'word_count': RunnableLambda(word_count)
    }
)

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic': 'AI'}))

{'joke': "Why did the artificial intelligence break up with her computer boyfriend? Because he couldn't handle her constant CTRL issues!", 'word_count': 19}


In [8]:
result = final_chain.invoke({'topic': 'AI'})
final_result = """{} \n word count - {}""".format(result['joke'], result['word_count'])
print(final_result)

Why did the AI break up with his girlfriend?

Because she couldn't handle his complex algorithms for love! 
 word count - 18


### RunnableBranch
- This is used for conditional chain implementation.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a detaailed report about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Summarize the following {text}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

report_gen_chain = RunnableSequence(prompt1, model, parser)

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>500, RunnableSequence(prompt2, model, parser)),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic': 'Russia vs Ukraine'}))

Russia and Ukraine have a long history of conflict and tension, dating back to ancient times. However, the current situation between the two countries is primarily rooted in the aftermath of the dissolution of the Soviet Union in 1991. Since then, the relationship between Russia and Ukraine has been fraught with territorial disputes, political disagreements, and military confrontations.

One of the key issues fueling the conflict between Russia and Ukraine is the status of Crimea, a region located in the southern part of Ukraine. In 2014, Russia annexed Crimea, following a controversial referendum that was widely condemned by the international community. This move was a violation of Ukraine's territorial integrity and sovereignty, leading to economic sanctions imposed by Western countries on Russia.

The annexation of Crimea was met with fierce opposition from Ukraine, which has since sought to regain control of the region. The conflict escalated into a full-blown war in eastern Ukrain

### LCEL
- Full form: `LangChain Execution Language`
- Calling `RunnableSequence` class via `| (pipe operator)` instead of `RunnableSequence` class. 
- This works for  `RunnableSequence` only as this is most useful runnable primitive among `RunnableSequence`, `RunnablePassthrough`, `RunnableParallel`, `RunnableLambda`, `RunnableBranch`.

Here let's see above example what we see in `RunnableBranch` again.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a detaailed report about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Summarize the following {text}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Till here it's same.
- Post here using `| (pipe operator)`.

In [2]:
report_gen_chain = prompt1 | model | parser 

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>500, prompt2 | model | parser),
    RunnablePassthrough()
)

final_chain = report_gen_chain | branch_chain

- Next, invoking final chain part will be same again.

In [3]:
print(final_chain.invoke({'topic': 'Russia vs Ukraine'}))

Introduction

The ongoing conflict between Russia and Ukraine has been a prominent issue in international politics for nearly a decade. The conflict began in 2014, following Russia's annexation of Crimea and its support for separatists in eastern Ukraine. Since then, tensions have remained high between the two countries, with sporadic ceasefire violations and continued fighting in eastern Ukraine.

Background

The roots of the conflict between Russia and Ukraine can be traced back to the collapse of the Soviet Union in 1991. Ukraine gained independence and began to distance itself from its former Soviet ally. In 2014, Ukraine's then-president, Viktor Yanukovych, was ousted from power following widespread protests against his decision to reject a trade deal with the European Union in favor of closer ties with Russia. This led to Russia's annexation of Crimea, a region with a significant Russian-speaking population, and its support for separatist movements in eastern Ukraine.

Legal and 